# Patient Feature Helpers (offset-aware, NBA 3.1)

Shared, data-source-agnostic implementation of the patient feature skill (`databricks-skills/patient_features_skills/SKILL.md`):
First Occurrence (Sec 3), Recency (Sec 4), Frequency A-D (Sec 5), Hospitalization A-D (Sec 6).

* Cohort logic comes ONLY from `00_cohort_config` (Sec 10.8): `assign_cohort_month`, `get_cohort_bounds`, `get_cohort_months`. The caller must `%run` the config BEFORE this notebook.
* `LOOKBACK_END_DATE` = cohort `cutoff`; `SVC_DT_M` = 1st-of-month cohort label; `COHORT` = `{YYYY-MM}_offset{D}`.
* Serverless-safe (Sec 11): no `.cache()/.persist()`, `F.count(F.lit(1))` inside pivots, sanitized column names, `overwriteSchema` on overwrite.

In [0]:
import re
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---- Default paths (Skill Sec 2.1) ----
BASE_QUERY_OUTPUT    = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/"
BASE_NOTEBOOK_OUTPUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/"
HCP_UNIVERSE_PATH    = BASE_NOTEBOOK_OUTPUT + "kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"

# ---- Tunable parameters (Skill Sec 9) ----
LOOKBACK_DURATION     = 365
RECENCY_TIME_WINDOWS  = [30, 60, 90, 180, 270, 360]
FREQ_LAG_WINDOWS      = [1, 2, 3, 6, 12]
FREQ_CHANGE_WINDOWS   = [1, 2, 3, 6]
FREQ_PATIENT_WINDOWS  = [30, 60, 90, 180, 360]
HOSP_LAG_WINDOWS      = [1, 6, 12]
HOSP_PATIENT_WINDOWS  = [30, 180, 360]
EVENT_PREVALENCE      = 0.00
RELEVANT_RX_EVENTS    = ["STEGLATRO", "INVOKANA", "KERENDIA"]
BATCH_SIZE            = 6

KEY_COLS = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT"]

# ---- Canonical schema (Skill Sec 1.3) ----
COLUMN_MAP = {
    "PROVIDER_ID": ["PROVIDER_ID", "HCP_ID", "PHYSICIAN_ID", "PROV_ID", "BHID"],
    "PATIENT_ID":  ["PATIENT_ID", "MEMBER_ID", "ENROLLEE_ID", "SUBSCRIBER_ID"],
    "EVENT_NAME":  ["PRODUCT_GROUP", "DIAGNOSIS_DESCRIPTION", "PROCEDURE_DESCRIPTION", "TEST_TYPE"],
    "EVENT_DATE":  ["SVC_DT", "SERVICE_DATE", "CLAIM_DATE", "FILL_DT"],
    "CLAIM_ID":    ["CLAIM_ID", "CLAIM_NBR", "CLAIM_NUMBER"],
}

def standardize(df, event_type):
    for canonical, variants in COLUMN_MAP.items():
        for v in variants:
            if v in df.columns and canonical not in df.columns:
                df = df.withColumnRenamed(v, canonical)
                break
    df = df.withColumn("EVENT_TYPE", F.lit(event_type))
    keep = [c for c in ["PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "EVENT_DATE",
                        "EVENT_TYPE", "CLAIM_ID", "DATA_SOURCE"] if c in df.columns]
    # EVENT_DATE is a string in some sources (e.g. PX) -> cast to date
    return df.select(*keep).withColumn("EVENT_DATE", F.to_date("EVENT_DATE"))

def sanitize_column_name(name):
    name = re.sub(r'[^A-Za-z0-9_]', '_', name.replace("+", "AND").replace("/", "_OR_"))
    name = re.sub(r'_+', '_', name).strip('_')
    return name.upper()

# ---- Market basket (Skill Sec 1.5) ----
# Baskets are defined next to the cohort config as RX_/DX_/PX_/LAB_MARKET_BASKET (Python lists).
# Skill Sec 1.5 TODO: final baskets not added yet -> fall back to all events in the input files.
MAX_UNMATCHED_BASKET_SHARE = 0.20

def get_market_basket(source):
    basket = globals().get(f"{source.upper()}_MARKET_BASKET")
    if not basket:
        print(f"WARNING: {source.upper()}_MARKET_BASKET not defined -> features for ALL events (Skill Sec 1.5 TODO)")
        return None
    print(f"{source.upper()} market basket: {len(basket)} events")
    return [e.strip().upper() for e in basket]

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

In [0]:
def output_root(test_hcp_sample):
    """All outputs of the offset run live in their own folder so production 3.0 outputs are never overwritten."""
    suffix = "_test" if test_hcp_sample and test_hcp_sample > 0 else ""
    return f"{BASE_NOTEBOOK_OUTPUT}patient_features_offset{OFFSET_DAY}{suffix}/"

def write_delta(df, path, partition_by=None, mode="overwrite"):
    w = df.write.format("delta").mode(mode)
    if mode == "overwrite":
        w = w.option("overwriteSchema", "true")
    if partition_by and partition_by in df.columns:   # Sec 11.3: only partition on existing columns
        w = w.partitionBy(partition_by)
    w.save(path)

def stage(df, path):
    """Replacement for cache() on serverless (Sec 7.4 / 11.1): write the intermediate and read it back."""
    write_delta(df, path)
    return spark.read.format("delta").load(path)

def load_hcp_universe(test_hcp_sample, stage_root):
    u = spark.read.format("delta").load(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
    if test_hcp_sample and test_hcp_sample > 0:
        # Deterministic HCP sample for test runs (config TEST_NPIS are NPIs, PROVIDER_ID is an IQVIA id)
        u = u.orderBy(F.xxhash64("PROVIDER_ID"), "PROVIDER_ID").limit(int(test_hcp_sample))
    return stage(u, stage_root + "hcp_universe")

In [0]:
def build_cohort_table(max_data_date):
    """One row per cohort (Skill Sec 2 + 10.8.3).
    COHORT = '{YYYY-MM}_offset{D}', SVC_DT_M = 1st of the reference month,
    LOOKBACK_END_DATE = cutoff (inclusive), LOOKBACK_START_DATE = cutoff - LOOKBACK_DURATION days.
    Only cohorts whose cutoff <= max_data_date are kept (complete windows)."""
    rows = []
    for m in get_cohort_months(max_data_date):
        cb = get_cohort_bounds(m)
        cutoff = cb["cutoff"].date() if isinstance(cb["cutoff"], datetime) else cb["cutoff"]
        rows.append((cb["cohort_id"], m, date.fromisoformat(f"{m}-01"), cutoff,
                     cutoff - timedelta(days=LOOKBACK_DURATION), int(cb["offset_days"])))
    if not rows:
        raise ValueError(f"No complete cohorts for max data date {max_data_date}")
    return spark.createDataFrame(
        rows, "COHORT string, COHORT_MONTH string, SVC_DT_M date, LOOKBACK_END_DATE date, "
              "LOOKBACK_START_DATE date, OFFSET_DAYS int")

def cohort_month_range(cohorts, lag_months):
    """Month spine: from (first cohort month - (lag_months-1)) to last cohort month."""
    r = cohorts.agg(F.min("SVC_DT_M").alias("mn"), F.max("SVC_DT_M").alias("mx")).collect()[0]
    from dateutil.relativedelta import relativedelta as _rd
    return r["mn"] - _rd(months=lag_months - 1), r["mx"]

In [0]:
def apply_market_basket(claims, basket):
    """Skill Sec 1.5: keep only basket events (broadcast semi-join on upper/trimmed EVENT_NAME)."""
    b = spark.createDataFrame([(e,) for e in basket], "BASKET_EVENT string")
    claims = claims.withColumn("BASKET_EVENT", F.upper(F.trim("EVENT_NAME")))
    unmatched = [r[0] for r in b.join(claims.select("BASKET_EVENT").distinct(), "BASKET_EVENT", "left_anti").collect()]
    print("Basket events with no claims:", unmatched)
    assert len(unmatched) <= MAX_UNMATCHED_BASKET_SHARE * len(basket), "Too many unmatched basket events"
    return claims.join(F.broadcast(b), "BASKET_EVENT", "left_semi").drop("BASKET_EVENT")

def load_claims(path, event_type, hcp_universe, max_cutoff, event_literal=None, basket=None):
    """Load -> standardize -> prune -> market basket -> keep HCP universe -> drop events after the last cutoff."""
    claims = (standardize(spark.read.format("delta").load(path), event_type)
              .filter(F.col("PROVIDER_ID").isNotNull() & F.col("EVENT_NAME").isNotNull()
                      & F.col("EVENT_DATE").isNotNull())
              .filter(F.col("EVENT_DATE") <= F.lit(max_cutoff))
              .join(F.broadcast(hcp_universe), "PROVIDER_ID", "left_semi"))
    if basket:                         # basket filter before any aggregation (Sec 1.5 rule 2)
        claims = apply_market_basket(claims, basket)
    if event_literal is not None:      # Hospitalization: collapse (basket) events (Sec 6, Sec 1.5 rule 5)
        claims = claims.withColumn("EVENT_NAME", F.lit(event_literal))
    return claims

def add_event_keys(claims, stage_path, basket=None, event_prevalence=EVENT_PREVALENCE, bypass=None):
    """Map each EVENT_NAME to a Delta-safe EVENT_KEY once (Sec 12.4). With a basket, the basket is the
    explicit pivot list (Sec 1.5 rule 3); without one, fall back to the prevalence filter (Sec 7.8).
    Returns (claims_with_key_staged, ordered list of keys)."""
    if basket:
        names = sorted(set(basket))
        match_col = F.upper(F.trim(F.col("EVENT_NAME")))
    else:
        bypass = [b.upper() for b in (bypass or [])]
        n_hcp = claims.select("PROVIDER_ID").distinct().count()
        prev = claims.groupBy("EVENT_NAME").agg(F.countDistinct("PROVIDER_ID").alias("n")).collect()
        names = sorted(r["EVENT_NAME"] for r in prev
                       if (n_hcp and r["n"] / n_hcp >= event_prevalence) or r["EVENT_NAME"].upper() in bypass)
        match_col = F.col("EVENT_NAME")
    mapping, used = [], {}
    for n in names:
        k = sanitize_column_name(n) or "UNKNOWN"
        if k in used:                       # disambiguate sanitization collisions
            used[k] += 1
            k = f"{k}_{used[k]}"
        else:
            used[k] = 1
        mapping.append((n, k))
    map_df = spark.createDataFrame(mapping, "MATCH_NAME string, EVENT_KEY string")
    staged = stage(claims.withColumn("MATCH_NAME", match_col)
                         .join(F.broadcast(map_df), "MATCH_NAME", "inner").drop("MATCH_NAME"), stage_path)
    keys = [k for _, k in mapping]
    print(f"{len(keys)} event keys ({'market basket' if basket else f'prevalence >= {event_prevalence}'})")
    return staged, keys

def expand_to_cohorts(claims, cohorts):
    """Attach every claim to each cohort whose 12M lookback contains it (Sec 4 12M filter)."""
    cb = cohorts.select("COHORT", "LOOKBACK_START_DATE", "LOOKBACK_END_DATE")
    mn = cohorts.agg(F.min("LOOKBACK_START_DATE")).collect()[0][0]
    return (claims.filter(F.col("EVENT_DATE") >= F.lit(mn))
            .join(F.broadcast(cb),
                  (F.col("EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
                  (F.col("EVENT_DATE") <= F.col("LOOKBACK_END_DATE")), "inner")
            .withColumn("EVENT_TIME", F.datediff("LOOKBACK_END_DATE", "EVENT_DATE"))
            .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))

def cohort_list(cohorts):
    return [r["COHORT"] for r in cohorts.select("COHORT").orderBy("COHORT").collect()]

In [0]:
def first_occurrence_features(claims, cohorts, event_keys, out_path, stage_root):
    """Skill Sec 3 (default order): Step 1 pre-aggregate min(EVENT_DATE) per (PROVIDER_ID, EVENT_KEY)
    -> Step 2 pivot the first-occurrence DATE wide per HCP (once, cohort-independent)
    -> Step 3 broadcast cross-join cohorts, EVENT_TIME = datediff(cutoff, date) per column,
       NULL when the first event is after the cutoff (leakage guard), drop all-NULL HCP-cohort rows.
    No cohort batching needed (output = N_HCP x N_cohorts rows)."""
    pre = stage(claims.groupBy("PROVIDER_ID", "EVENT_KEY")
                      .agg(F.min("EVENT_DATE").alias("FIRST_OCCURRENCE_DATE")),
                stage_root + "first_occ_pre")                                          # Step 1
    wide = stage(pre.groupBy("PROVIDER_ID").pivot("EVENT_KEY", event_keys)
                    .agg(F.min("FIRST_OCCURRENCE_DATE")),
                 stage_root + "first_occ_wide")                                        # Step 2
    fo_cols = [f"MAX_EVENT_TIME_{k}_FIRST_OCCURRENCE" for k in event_keys]
    any_event = (F.greatest(*[F.col(c).isNotNull().cast("int") for c in fo_cols]) == 1
                 if len(fo_cols) > 1 else F.col(fo_cols[0]).isNotNull())
    out = (wide.crossJoin(F.broadcast(cohorts.select("COHORT", "LOOKBACK_END_DATE")))  # Step 3
               .select("PROVIDER_ID", "COHORT",
                       *[F.when(F.datediff("LOOKBACK_END_DATE", F.col(k)) >= 0,
                                F.datediff("LOOKBACK_END_DATE", F.col(k))).alias(c)
                         for k, c in zip(event_keys, fo_cols)])
               .filter(any_event)                                  # drop all-NULL HCP-cohort rows
               .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))
    write_delta(out.select(*KEY_COLS, *fo_cols), out_path, "COHORT")
    print(f"First occurrence -> {out_path}")

In [0]:
def recency_features(claims, cohorts, event_keys, event_types, out_path, stage_root,
                     windows=RECENCY_TIME_WINDOWS):
    """Skill Sec 4: 12M-filtered claims, six sub-features joined on HCP_COHORT_ID."""
    for i, batch in enumerate(chunks(cohort_list(cohorts), BATCH_SIZE)):
        c12 = stage(expand_to_cohorts(claims, cohorts.filter(F.col("COHORT").isin(batch)))
                    .select(*KEY_COLS, "EVENT_KEY", "EVENT_TYPE", "EVENT_TIME"),
                    stage_root + "recency_c12")
        # Sub 1: most recent event per event
        s1 = (c12.groupBy("HCP_COHORT_ID").pivot("EVENT_KEY", event_keys).agg(F.min("EVENT_TIME"))
                 .select("HCP_COHORT_ID", *[F.col(k).alias(sanitize_column_name(f"min_EVENT_TIME_{k}_time_min"))
                                            for k in event_keys]))
        # Sub 2-6 in a single aggregation pass
        aggs = [F.count(F.lit(1)).alias("total_events"),
                F.countDistinct("EVENT_KEY").alias("unique_events")]
        for n in windows:
            in_w = F.col("EVENT_TIME") <= n
            aggs += [F.count(F.when(in_w, 1)).alias(f"total_events_time_{n}_days"),
                     F.countDistinct(F.when(in_w, F.col("EVENT_KEY"))).alias(f"unique_events_time_{n}_days")]
        for t in event_types:
            is_t = F.col("EVENT_TYPE") == t
            aggs += [F.count(F.when(is_t, 1)).alias(f"total_{t}"),
                     F.countDistinct(F.when(is_t, F.col("EVENT_KEY"))).alias(f"unique_{t}")]
            for n in windows:
                cond = is_t & (F.col("EVENT_TIME") <= n)
                aggs += [F.count(F.when(cond, 1)).alias(f"total_events_time_{n}_days_{t}"),
                         F.countDistinct(F.when(cond, F.col("EVENT_KEY"))).alias(f"unique_events_time_{n}_days_{t}"),
                         # NULL when < 3 events in the slice (Sec 4, sub-feature 6)
                         F.when(F.count(F.when(cond, 1)) >= 3,
                                F.skewness(F.when(cond, F.col("EVENT_TIME")))).alias(f"skewness_time_{n}_days_{t}")]
        s26 = c12.groupBy(*KEY_COLS).agg(*aggs)
        write_delta(s26.join(s1, "HCP_COHORT_ID", "left"), out_path, "COHORT", "overwrite" if i == 0 else "append")
    print(f"Recency -> {out_path}")

In [0]:
def fill_missing(df, value_cols, min_month, max_month):
    """Dense PROVIDER_ID x cohort-month spine (Sec 5 Sub-B, 10.8.3). Months are 1st-of-month cohort labels."""
    months = spark.sql(f"SELECT explode(sequence(to_date('{min_month}'), to_date('{max_month}'), "
                       f"interval 1 month)) AS SVC_DT_M")
    spine = df.select("PROVIDER_ID").distinct().crossJoin(F.broadcast(months))
    return spine.join(df, ["PROVIDER_ID", "SVC_DT_M"], "left").fillna(0, subset=value_cols)

def add_lags(df, value_cols, windows):
    """Rolling sums over the last N cohort-months (rowsBetween on the dense spine)."""
    w = Window.partitionBy("PROVIDER_ID").orderBy("SVC_DT_M")
    return df.select("PROVIDER_ID", "SVC_DT_M",
                     *[F.sum(F.col(c)).over(w.rowsBetween(-i + 1, 0)).alias(f"{c}_IN_LAST_{i}_MONTH")
                       for c in value_cols for i in windows])

def to_cohort_grain(df, cohorts):
    """Keep only cohort months and attach COHORT / HCP_COHORT_ID (Sec 8.1)."""
    return (df.join(F.broadcast(cohorts.select("COHORT", "SVC_DT_M")), "SVC_DT_M", "inner")
              .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))

In [0]:
def frequency_features(claims, cohorts, event_keys, patient_label, out_root, source, stage_root):
    """Skill Sec 5. patient_label e.g. 'NUM_OF_PATIENTS_THE_HCP_PRESCRIBED_WITH_{event}_IN_LAST_{n}_DAYS'."""
    min_m, max_m = cohort_month_range(cohorts, max(FREQ_LAG_WINDOWS))
    freq_cols = [f"FREQ_{k}" for k in event_keys]

    # Sub-A: monthly claim count (SVC_DT_M via assign_cohort_month, never date_trunc)
    a = (claims.withColumn("SVC_DT_M", assign_cohort_month(F.col("EVENT_DATE")))
               .filter((F.col("SVC_DT_M") >= F.lit(min_m)) & (F.col("SVC_DT_M") <= F.lit(max_m)))
               .groupBy("PROVIDER_ID", "SVC_DT_M").pivot("EVENT_KEY", event_keys).agg(F.count(F.lit(1)))
               .select("PROVIDER_ID", "SVC_DT_M", *[F.col(k).alias(f"FREQ_{k}") for k in event_keys])
               .fillna(0, subset=freq_cols))
    a = stage(a, out_root + f"{source}_frequency_monthly_claim_count_features")

    # Sub-B: lag frequency on dense spine
    filled = stage(fill_missing(a, freq_cols, min_m, max_m), stage_root + "freq_filled")
    lag = to_cohort_grain(add_lags(filled, freq_cols, FREQ_LAG_WINDOWS), cohorts)
    lag = stage(lag, out_root + f"{source}_frequency_lag_features")

    # Sub-C: change in frequency vs own 12M baseline
    chg = [((F.col(f"FREQ_{k}_IN_LAST_{n}_MONTH") / n) -
            ((F.col(f"FREQ_{k}_IN_LAST_12_MONTH") - F.col(f"FREQ_{k}_IN_LAST_{n}_MONTH")) / (12 - n)))
           .alias(f"CHANGE_IN_FREQ_OF_HCP_CLAIMS_FOR_{k}_IN_LAST_{n}_MONTH")
           for k in event_keys for n in FREQ_CHANGE_WINDOWS]
    write_delta(lag.select(*KEY_COLS, "SVC_DT_M", *chg),
                out_root + f"{source}_frequency_change_features", "COHORT")

    # Sub-D: unique patients per HCP x cohort x event x window
    for i, batch in enumerate(chunks(cohort_list(cohorts), BATCH_SIZE)):
        c = expand_to_cohorts(claims, cohorts.filter(F.col("COHORT").isin(batch)))
        agg = (c.select("PROVIDER_ID", "COHORT", "EVENT_KEY", "PATIENT_ID", "EVENT_TIME")
                .groupBy("PROVIDER_ID", "COHORT", "EVENT_KEY")
                .agg(*[F.countDistinct(F.when(F.col("EVENT_TIME") <= n, F.col("PATIENT_ID"))).alias(f"W{n}")
                       for n in FREQ_PATIENT_WINDOWS]))
        p = (agg.groupBy("PROVIDER_ID", "COHORT").pivot("EVENT_KEY", event_keys)
                .agg(*[F.first(f"W{n}").alias(f"W{n}") for n in FREQ_PATIENT_WINDOWS]))
        cols = [F.col(f"{k}_W{n}").alias(patient_label.format(event=k, n=n))
                for k in event_keys for n in FREQ_PATIENT_WINDOWS]
        d = (p.select("PROVIDER_ID", "COHORT", *cols)
              .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT"))
              .fillna(0))
        write_delta(d.select(*KEY_COLS, *[c_ for c_ in d.columns if c_ not in KEY_COLS]),
                    out_root + f"{source}_frequency_unique_patient_features", "COHORT",
                    "overwrite" if i == 0 else "append")
    print(f"Frequency A-D -> {out_root}{source}_frequency_*")

In [0]:
def hospitalization_features(total_claims, hosp_claims, cohorts, hcp_universe, event_literal,
                             hosp_patient_label, out_root, source, stage_root, col_suffix=""):
    """Skill Sec 6. Claims must already carry EVENT_NAME = event_literal (ALL_DIAGNOSIS / ALL_PROCEDURES).
    hosp_claims are pre-filtered upstream (no DATA_SOURCE filter, Sec 11.5)."""
    min_m, max_m = cohort_month_range(cohorts, max(HOSP_LAG_WINDOWS))
    tot_c, hos_c = f"TOTAL_CLAIMS_{event_literal}", f"TOTAL_HOSPITALIZED_CLAIMS_{event_literal}"

    def _monthly(df, name):
        return (df.withColumn("SVC_DT_M", assign_cohort_month(F.col("EVENT_DATE")))
                  .filter((F.col("SVC_DT_M") >= F.lit(min_m)) & (F.col("SVC_DT_M") <= F.lit(max_m)))
                  .groupBy("PROVIDER_ID", "SVC_DT_M").agg(F.count(F.lit(1)).alias(name)))

    # Sub-A: total + hospitalized monthly counts
    a = _monthly(total_claims, tot_c).join(_monthly(hosp_claims, hos_c), ["PROVIDER_ID", "SVC_DT_M"], "left").fillna(0)
    a = stage(a, out_root + f"{source}_hospitalization_monthly_counts")

    # Sub-B: lag on dense spine + validation filter (TOTAL >= HOSPITALIZED)
    filled = stage(fill_missing(a, [tot_c, hos_c], min_m, max_m), stage_root + "hosp_filled")
    lag = add_lags(filled, [tot_c, hos_c], HOSP_LAG_WINDOWS)
    for n in HOSP_LAG_WINDOWS:
        lag = lag.filter(F.col(f"{tot_c}_IN_LAST_{n}_MONTH") >= F.col(f"{hos_c}_IN_LAST_{n}_MONTH"))
    lag = to_cohort_grain(lag, cohorts)

    # Sub-C: percent hospitalized; drop TOTAL_CLAIMS_* lags, keep PERC_* and TOTAL_HOSPITALIZED_*
    perc = [F.when(F.col(f"{tot_c}_IN_LAST_{n}_MONTH") != 0,
                   F.col(f"{hos_c}_IN_LAST_{n}_MONTH") / F.col(f"{tot_c}_IN_LAST_{n}_MONTH"))
             .alias(f"PERC_HOSPITALIZED_HCP_CLAIMS_FOR_{event_literal}_IN_LAST_{n}_MONTH") for n in HOSP_LAG_WINDOWS]
    c = stage(lag.select("PROVIDER_ID", "SVC_DT_M", *perc,
                         *[f"{hos_c}_IN_LAST_{n}_MONTH" for n in HOSP_LAG_WINDOWS]),
              stage_root + "hosp_perc")

    # Sub-D: unique hospitalized patients per HCP x cohort
    d = (expand_to_cohorts(hosp_claims, cohorts)
           .groupBy("PROVIDER_ID", "COHORT")
           .agg(*[F.countDistinct(F.when(F.col("EVENT_TIME") <= n, F.col("PATIENT_ID")))
                  .alias(hosp_patient_label.format(n=n)) for n in HOSP_PATIENT_WINDOWS]))

    # Combine Everything: skeleton = HCP universe x cohorts
    skeleton = (hcp_universe.crossJoin(F.broadcast(cohorts.select("COHORT", "SVC_DT_M")))
                .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))
    final = (skeleton.join(c, ["PROVIDER_ID", "SVC_DT_M"], "left")
                     .join(d, ["PROVIDER_ID", "COHORT"], "left"))
    feat = [x for x in final.columns if x not in KEY_COLS + ["SVC_DT_M"]]
    final = final.select(*KEY_COLS, "SVC_DT_M", *[F.col(x).alias(f"{x}{col_suffix}") for x in feat])
    write_delta(final, out_root + f"{source}_hospitalization_features", "COHORT")
    print(f"Hospitalization -> {out_root}{source}_hospitalization_features")

In [0]:
def validate_outputs(out_root, source, claims=None):
    """Key uniqueness + offset checks (Skill Sec 10.8.6) for every output of a source."""
    paths = [p.path for p in dbutils.fs.ls(out_root) if p.name.rstrip("/").startswith(f"{source}_")]
    ok = True
    for p in sorted(paths):
        df = spark.read.format("delta").load(p)
        n = df.count()
        msg = f"{p.rstrip('/').split('/')[-1]:<55} rows={n:>10,} cols={len(df.columns):>5}"
        if "HCP_COHORT_ID" in df.columns:
            dup = n - df.select("HCP_COHORT_ID").distinct().count()
            off = [r[0] for r in df.select(F.regexp_extract("COHORT", r"_offset(\d+)$", 1)).distinct().collect()]
            msg += f" dup_keys={dup} offsets={off}"
            ok &= (dup == 0) and off == [str(OFFSET_DAY)]
        print(msg)
    if claims is not None:   # events on days 1..D-1 must fall in the PREVIOUS cohort month
        bad = (claims.filter((F.dayofmonth("EVENT_DATE") < OFFSET_DAY) & (F.dayofmonth("EVENT_DATE") >= 1))
                     .filter(assign_cohort_month(F.col("EVENT_DATE")) == F.trunc("EVENT_DATE", "month")).count())
        print(f"Offset bucketing check (days < {OFFSET_DAY} mapped to previous month): {'PASS' if bad == 0 else f'FAIL {bad}'}")
        ok &= bad == 0 or OFFSET_DAY <= 1
    assert ok, "Validation failed"
    print("All validation checks PASSED")